In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from pathlib import Path
import os
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from sklearn.linear_model import LinearRegression
from scipy.stats import t as t_dist   

from train_utils import get_info
from model import DiagnosticModel
from data import DicomDataset, split_dataset

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# DataFrame Metrics

In [3]:
folders = [Path("../../outputs_stack_exp")]

In [4]:
all_experiments = []

for base in folders:
    for exp_name in os.listdir(base):
        exp_path = base / exp_name
        if not exp_path.is_dir():
            continue

        params_path  = exp_path / "params.json"

        for run_name in os.listdir(exp_path):
            run_path = exp_path / run_name
            if not run_path.is_dir():
                continue

            # model_path   = run_path / "best_model.pth"
            results_path = run_path / "test_results.json"

            if not results_path.exists():
                continue

            with open(params_path, "r") as f:
                params = json.load(f)

            with open(results_path, "r") as f:
                results = json.load(f)

            with open(run_path / 'info.json') as f:
                run_info = json.load(f)

            entry = {
                "exp": exp_name,
                "run": run_name,
                "auc": float(results["auc"]),
                "train_pos": run_info['train_cnt']['pos'],
                "train_neg": run_info['train_cnt']['neg'],
                "val_pos": run_info['val_cnt']['pos'],
                "val_neg": run_info['val_cnt']['neg'],
                "test_pos": run_info['test_cnt']['pos'],
                "test_neg": run_info['test_cnt']['neg'],
            }

            # Add confusion-matrix-derived metrics
            for k, v in get_info(np.array(results["val_cfvalues"])).items():
                entry[k] = v

            all_experiments.append(entry)


/data/vision/polina/users/marcusbl/bin_class/src/train_utils.py:105: RuntimeWarning: invalid value encountered in scalar divide
  info['prec'] = info['tp'] / (info['tp'] + info['fp'])


In [5]:
df = pd.DataFrame(all_experiments)

# Convert the meta columns into index
df = df.set_index(["exp", "run"]).sort_index()
pd.set_option('display.max_rows', None)
df

auc  train_pos  train_neg  val_pos  val_neg  \
exp                run                                                      
mm_2stack          run0  0.900832        851       4391      326     1109   
                   run1  0.910126        811       3610      226     1238   
                   run2  0.874317        891       3407      246     1312   
                   run3  0.844768        963       3688      154      957   
                   run4  0.909030        984       4116      225      884   
                   run5  0.875498        952       4616      186      457   
mm_2stack2_untrain run0  0.864186       1033       4580      183      867   
                   run1  0.875130        911       4076      269     1014   
                   run2  0.484303        872       4002      222      941   
                   run3  0.869702        707       3606      434     1410   
                   run4  0.908294        821       3332      108     1215   
                   run5  0.898045       1108       4232      147      510   
mm_2stack_untrain  run0  0.908996        851       4391      326     1109   
                   run1  0.531834        811       3610      226     1238   
                   run2  0.876683        891       3407      246     1312   
                   run3  0.844275        963       3688      154      957   
                   run4  0.913668        984       4116      225      884   
                   run5  0.864867        952       4616      186      457   
mm_none            run0  0.908002        851       4391      326     1109   
                   run1  0.910974        811       3610      226     1238   
                   run2  0.887231        891       3407      246     1312   
                   run3  0.832413        963       3688      154      957   
                   run4  0.885899        984       4116      225      884   
                   run5  0.870103        952       4616      186      457   
mm_none2           run0  0.871128       1033       4580      183      867   
                   run1  0.868266        911       4076      269     1014   
                   run2  0.813377        872       4002      222      941   
                   run3  0.873515        707       3606      434     1410   
                   run4  0.907319        821       3332      108     1215   
                   run5  0.927854       1108       4232      147      510   
mm_none2_untrain   run0  0.822656       1033       4580      183      867   
                   run1  0.868276        911       4076      269     1014   
                   run2  0.522697        872       4002      222      941   
                   run3  0.848551        707       3606      434     1410   
                   run4  0.893206        821       3332      108     1215   
                   run5  0.893690       1108       4232      147      510   
mm_none_untrain    run0  0.468583        851       4391      326     1109   
                   run1  0.889507        811       3610      226     1238   
                   run2  0.845860        891       3407      246     1312   
                   run3  0.822274        963       3688      154      957   
                   run4  0.884447        984       4116      225      884   
                   run5  0.847976        952       4616      186      457   
mm_stack           run0  0.936996        851       4391      326     1109   
                   run1  0.931995        811       3610      226     1238   
                   run2  0.885354        891       3407      246     1312   
                   run3  0.845854        963       3688      154      957   
                   run4  0.901474        984       4116      225      884   
                   run5  0.873024        952       4616      186      457   
mm_stack2          run0  0.894631       1033       4580      183      867   
                   run1  0.883837        911       4076      269     1014   
                   run2  0

In [6]:
exp_avg = df.groupby("exp").mean().sort_values(by='auc', ascending=False)
exp_avg

,auc,train_pos,train_neg,val_pos,val_neg,test_pos,test_neg,tn,fp,fn,tp,tpr,fpr,prec,recall,f1,acc
exp,,,,,,,,,,,,,,,,,
mm_stack,0.895783,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.716726,0.088239,0.050977,0.144058,0.723054,0.109091,0.658586,0.723054,0.669477,0.860785
mm_2stack,0.885762,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.712365,0.092599,0.056640,0.138395,0.692552,0.116098,0.626647,0.692552,0.642695,0.850760
mm_none,0.882437,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.723933,0.081031,0.057926,0.137109,0.685005,0.100407,0.645115,0.685005,0.651492,0.861042
mm_stack2,0.877238,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.718668,0.095403,0.053715,0.132213,0.696381,0.119845,0.610907,0.696381,0.640889,0.850882
mm_none2,0.876910,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.735568,0.078504,0.060344,0.125585,0.671809,0.098542,0.646646,0.671809,0.650061,0.861153
mm_stack2_untrain,0.865633,1033.000000,4580.000000,183.000000,867.000000,147.000000,510.000000,0.643836,0.132420,0.051750,0.171994,0.768707,0.170588,0.565000,0.768707,0.651297,0.815830
mm_2stack_untrain,0.823387,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.722584,0.082381,0.086810,0.108225,0.561627,0.101545,0.574344,0.561627,0.612283,0.830809
mm_2stack2_untrain,0.816610,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.732114,0.081958,0.078784,0.107144,0.569454,0.103599,0.653552,0.569454,0.519316,0.839258
mm_none2_untrain,0.808179,908.666667,3971.333333,227.166667,992.833333,227.166667,992.833333,0.723203,0.090868,0.082685,0.103243,0.555129,0.113526,0.521972,0.555129,0.582791,0.826446


## Explore Effect of Dataset Distributions on AUC

We attempt to fit a multi-variate regression of AUC vs. {train_pos_ratio, val_pos_ratio, test_pos_ratio} to see if the pos_ratio of the datasets affects the final AUC

In [7]:
def regression_for_experiment(exp: str, gdf: pd.DataFrame):
    """
    gdf = all runs belonging to ONE experiment (exp)
    """
    # --- Create predictors correctly ---
    # x_train = np.array(gdf['train_pos'] + gdf['train_neg'], dtype=np.float32)
    # x_val   = np.array(gdf['val_pos']   + gdf['val_neg'], dtype=np.float32)
    # x_test  = np.array(gdf['test_pos']  + gdf['test_neg'], dtype=np.float32)

    x_train = np.array(gdf['train_pos'] / (gdf['train_pos'] + gdf['train_neg']), dtype=np.float32)
    x_val   = np.array(gdf['val_pos']   / (gdf['val_pos']   + gdf['val_neg']), dtype=np.float32)
    x_test  = np.array(gdf['test_pos']  / (gdf['test_pos']  + gdf['test_neg']), dtype=np.float32)

    X = np.column_stack([x_train, x_val, x_test])
    y = np.array(gdf['auc'], dtype=float)
    
    # Add intercept manually
    X_design = np.column_stack([np.ones(len(X)), X])

    n, p = X_design.shape
    df = n - p

    if df <= 0:
        return None  # underdetermined

    # --- Fit OLS in sklearn ---
    reg = LinearRegression(fit_intercept=False).fit(X_design, y)
    beta = reg.coef_
    y_pred = reg.predict(X_design)

    # --- Stats ---
    resid = y - y_pred
    sigma2 = (resid @ resid) / df
    cov_beta = sigma2 * np.linalg.inv(X_design.T @ X_design)
    se_beta = np.sqrt(np.diag(cov_beta))
    t_stats = beta / se_beta
    p_vals = 2 * (1 - t_dist.cdf(np.abs(t_stats), df=df))

    # -----------------------------------------
    #  FIRST PLOT (unchanged)
    # -----------------------------------------
    plt.figure(figsize=(7,5))

    predictors = {
        "train fraction pos": x_train,
        "val fraction pos":   x_val,
        "test fraction pos":  x_test,
    }

    colors = {
        "train fraction pos": "red",
        "val fraction pos": "blue",
        "test fraction pos": "green",
    }

    for name, vals in predictors.items():
        plt.scatter(y, vals, alpha=0.65, label=name, color=colors[name])

    plt.xlabel("AUC")
    plt.ylabel("Fraction positives")
    plt.title(f"Predictors vs AUC ({exp})")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # -----------------------------------------
    #  SECOND PLOT: 3 SUBPLOTS (AUC vs each x)
    # -----------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    pred_list = [
        ("train fraction pos", x_train),
        ("val fraction pos",   x_val),
        ("test fraction pos",  x_test),
    ]

    for ax, (name, vals) in zip(axes, pred_list):
        ax.scatter(vals, y, alpha=0.7, color=colors[name])
        ax.set_title(f"{name} vs AUC")
        ax.set_xlabel(name)
        ax.set_ylabel("AUC")

    fig.suptitle(f"Predictors vs AUC (subplots) — {exp}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Return stats unchanged
    return {
        "beta": beta,
        "se": se_beta,
        "t": t_stats,
        "p": p_vals,
        "df": df,
        "n": n
    }


In [ ]:
results = {}

for exp, gdf_exp in df.groupby("exp"):
    results[exp] = regression_for_experiment(exp, gdf_exp)

# Convert to a clean DataFrame
rows = []
for exp, res in results.items():
    if res is None:
        continue
    beta = res["beta"]
    se = res["se"]
    t = res["t"]
    p = res["p"]

    rows.append({
        "exp": exp,
        "p_intercept": p[0],
        "p_train": p[1],
        "p_val": p[2],
        "p_test": p[3],
        "beta_intercept": beta[0],
        "beta_train": beta[1],
        "beta_val": beta[2],
        "beta_test": beta[3],
        "se_intercept": se[0],
        "se_train": se[1],
        "se_val": se[2],
        "se_test": se[3],
        "t_intercept": t[0],
        "t_train": t[1],
        "t_val": t[2],
        "t_test": t[3],
        "df": res["df"],
        "n": res["n"],
    })

regression_summary = pd.DataFrame(rows).set_index("exp")
regression_summary


# Sort Model Results

## Set Variables

In [7]:
run_dir = Path('../../outputs_stack_exp/mm_stack/run0')

# Model Info
best_model_path = run_dir / 'best_model.pth'

# Data Info
info_path = run_dir / 'info.json'
data_distr = "test" # train, val, test, or full
data_dir = Path('/data/vision/polina/users/marcusbl/data')
batch_size = 16

# Output Info
out_dir = Path('../../outputs0')
out_dir.mkdir(exist_ok=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Get Model

In [9]:
model = DiagnosticModel(model_name = 'resnet50', include_weights = False, in_channels = 2)

checkpoint = torch.load(best_model_path, map_location=next(model.parameters()).device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.to(device)
''

''

## Get Data

In [10]:
dataset = DicomDataset(data_dir)

with open(info_path, 'r') as f:
    info_dict = json.load(f)

people = list(info_dict['ppl_ids'].values())
print(people)

train_data, val_data, test_data = split_dataset(dataset, people)

datasets = {
    "train": train_data,
    "val": val_data, 
    "test": test_data,
}

dataloaders = {} 
for name, dataset in datasets.items():
    dataset.set_norm(norm_method = 'min-max', masked_norm = True, perc_norm = 0.02, mask_method='stack')
    dataloaders[name] = DataLoader(dataset, shuffle = False, batch_size = batch_size)

Loading People Data: 100%|██████████| 30/30 [00:00<00:00, 33.73it/s]

[[23, 4, 2, 25, 6, 18, 13, 7, 27, 1, 16, 0, 15, 29, 28, 9, 8, 12, 11, 5], [20, 26, 3, 24, 22], [17, 21, 10, 19, 14]]


##  Run the Model on Data (~2 min on cuda for full dataset)

In [11]:
def get_confidence(outputs: torch.Tensor, method: str = "prob"):
    """
    outputs of model is (B, 2). confidence is (B,) and has values [0,1]
    
    There are two ways to calculate confidence:

    1) prob: Take prob (using softmax) of the predicted class
    2) odds: Take log-odds (z_1 - z_0) and then sigmoid it; do 1-answer if predicting 0
    """
    _, preds = torch.max(outputs, dim=1)

    if method == "prob":
        probs = torch.softmax(outputs, dim=1)
        conf = probs[torch.arange(len(probs)), preds]
    else:
        ratio = outputs[:, 1] - outputs[:, 0]
        base = torch.sigmoid(ratio)
        conf = torch.where(preds == 1, base, 1 - base)

    return conf


In [12]:
all_dfs = []

for split_name, dataloader in dataloaders.items():
    results = {
        "conf_prob": [],
        # "conf_odds": [],
        "idx": [],
        "pred": [],
        "true": [],
        "split": [],
    }

    for batch_idx, (data, labels) in enumerate(tqdm(dataloader, desc=split_name)):
        data, labels = data.to(device), labels.to(device)
        outputs = model(data) # (B, 2)
        _, preds = torch.max(outputs, dim=1)

        # Calculate Confidence
        conf_prob = get_confidence(outputs, method = 'prob')
        # conf_odds = get_confidence(outputs, method = 'odds')

        results['conf_prob'].extend(conf_prob.tolist())
        # results['conf_odds'].extend(conf_odds.tolist())

        results['idx'].extend([batch_idx * batch_size + i for i in range(len(labels))])
        results['pred'].extend(preds.tolist())
        results['true'].extend(labels.tolist())
        results['split'].extend([split_name] * len(labels))  # tag the split

    # Create a DataFrame for just this split
    split_df = pd.DataFrame(results)
    all_dfs.append(split_df)

# Combine all splits into one dataframe
results_df = pd.concat(all_dfs, ignore_index=True)

test: 100%|██████████| 41/41 [00:15<00:00,  2.66it/s]


In [13]:
results_df.head()

,conf_prob,idx,pred,true,split
0,0.985489,0,1,1,train
1,0.807157,1,1,1,train
2,0.750790,2,1,1,train
3,0.914253,3,1,1,train
4,0.644294,4,1,0,train


## Display results by confidence 

In [14]:
def display_hist(df: pd.DataFrame, fpath: Path, method: str):

    fig, ax = plt.subplots()
    df[f"conf_{method}"].hist(ax=ax)
    ax.set_title("Confidence Histogram")

    fig.savefig(fpath / "hist.png")
    plt.close(fig)


In [15]:
def display_preds(df: pd.DataFrame, dataset: DicomDataset, norm_params: None | dict = None,
                  max_display: int = 10, fpath: Path | None = None, top: bool = False, method: str = 'prob'):
    
        
    fpath.mkdir(exist_ok=True, parents=True)
    df = df.sort_values(by=f'conf_{method}', ascending=(not top)).reset_index(drop=True)

    # Fix Normalization
    if norm_params is not None:
        dataset.set_norm(
            norm_method = norm_params['norm_method'],
            masked_norm = norm_params['masked_norm'],
            perc_norm   = norm_params['perc_norm']
        )

    # Grid size
    num_display = min(len(df), max_display)
    nrows = int(np.sqrt(num_display))
    ncols = int(np.ceil(num_display / nrows))

    # Correct figsize scaling
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5 * ncols, 3 * nrows),
        squeeze=False
    )

    # Display images
    for i in range(nrows):
        for j in range(ncols):
            df_row_idx = i * ncols + j
            ax = axes[i, j]

            # Blank cell if overflow
            if df_row_idx >= num_display:
                ax.axis("off")
                continue

            row = df.iloc[df_row_idx]
            display_idx = int(row["idx"])
            pred = row["pred"]
            true = row["true"]
            conf = row[f"conf_{method}"]

            # Assuming dataset[idx] returns (img_tensor, label)
            img = dataset[display_idx][0][0]

            ax.imshow(img, cmap="gray", vmin=0, vmax = 1)
            ax.axis("off")

            title = f"True: {true} | Pred: {pred} (Idx {display_idx})"
            subtitle = f"Confidence: {conf:.3f}" if conf is not None else ""

            ax.set_title(title, fontsize=14, pad=10)
            ax.text(
                0.5, -0.12, subtitle,
                transform=ax.transAxes,
                ha="center", va="top", fontsize=12
            )

    fig.tight_layout()

    # Save or show
    if fpath is not None:
        if top:
            fig.savefig(fpath / "examples_top.png")
        else:
            fig.savefig(fpath / "examples_bottom.png")
    else:
        plt.show()

    plt.close(fig)


In [16]:
mm_norm_params = {
    'norm_method': 'min-max',
    'masked_norm': True,
    'perc_norm': 0,

}

masks = {
    "fp": (results_df['pred'] == 1) & (results_df['true'] == 0),
    "tp": (results_df['pred'] == 1) & (results_df['true'] == 1),
    "fn": (results_df['pred'] == 0) & (results_df['true'] == 1),
    "tn": (results_df['pred'] == 0) & (results_df['true'] == 0)
}

# prob = odds method!!!
# for method in ['prob', 'odds']:
for split in ['train', 'val', 'test']:
    split_mask = (results_df['split'] == split)
    for name, type_mask in masks.items():
        out_sbdir = out_dir  / split / name
        display_preds(results_df[type_mask & split_mask], datasets[split], mm_norm_params, max_display = 16, fpath = out_sbdir, top = True, method = 'prob')
        display_preds(results_df[type_mask & split_mask], datasets[split], mm_norm_params, max_display = 16, fpath = out_sbdir, top = False, method = 'prob')
        display_hist(results_df[type_mask & split_mask], fpath = out_sbdir, method = 'prob')